# ParseCrate: Document Processing API - Google Colab Demo

Convert a sample PDF into **Markdown and structured JSON**. Select
**Runtime → Run all** to try the complete example. A standard **CPU**
runtime is sufficient; no external API key is required.
Allow about five minutes for initial setup.

[ParseCrate website](https://parsecrate.com/) ·
[GitHub and Docker setup](https://github.com/hwdsl2/parsecrate)

This notebook runs the `docling-serve` API used by ParseCrate directly
in Colab, with local document models and `X-Api-Key` authentication.
Docker deployment and management commands are documented on GitHub.

**Privacy:** Documents are processed in your Google Colab runtime.
Use documents you have permission to upload. Colab's data practices
apply. For processing on your own infrastructure, use ParseCrate's
Docker deployment. Model/package downloads require internet access.

## 1. Prepare the demo

Run the setup cells below. Select **Show code** to expand a collapsed
cell. Dependencies are installed in section 2.

Setup downloads two original sample documents from ParseCrate's
website and checks their expected sizes and SHA-256 checksums:

- [Download the sample PDF](https://parsecrate.com/demo/parsecrate-demo.pdf)
- [View the scanned-page image](https://parsecrate.com/demo/parsecrate-scan.png)

Verified sample files are cached for subsequent setup runs.

Use a standard Linux x86_64 Colab runtime with **Python 3.12 or 3.13**.
The server has its own Python environment; Colab's ML packages are
left intact. Change `API_PORT` only if the default is already occupied.

In [ ]:
from pathlib import Path

API_PORT = 5001
DEMO_ROOT = Path("/content/parsecrate-demo")

# Release only the server owned by the previous run of this notebook.
if globals().get("parse_demo") is not None:
    parse_demo.stop_server()
parse_demo = None
parse_api_key = None
sample_path = selected_path = None

In [ ]:
# @title Set up the API helpers { display-mode: "form" }
"""Runtime helpers for the ParseCrate Colab demo.

Copyright (C) 2026 Lin Song. Licensed under the project's MIT License.
"""

from __future__ import annotations

import hashlib
import http.client
import json
import mimetypes
import os
from pathlib import Path
import platform
import re
import secrets
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import time


class DemoError(RuntimeError):
    """An actionable demo failure without credentials or a request-body dump."""


class ParseCrateDemo:
    MAX_FILE_BYTES = 20 * 1024 * 1024
    MAX_RESPONSE_BYTES = 32 * 1024 * 1024
    TERMINAL = {"success", "partial_success", "failure", "skipped"}
    PENDING = {"pending", "started"}

    def __init__(self, root: Path, port: int = 5001):
        if type(port) is not int or not 1024 <= port <= 65535:
            raise DemoError("Choose an unprivileged API port between 1024 and 65535.")
        self.root = Path(root).resolve()
        self.port = port
        self.process: subprocess.Popen | None = None
        self.key = ""
        self._redaction_keys: set[str] = set()
        self.task_id: str | None = None
        self.submission_uncertain = False
        self.task_kind = "convert"
        self.active_output: Path | None = None
        self.active_response: dict | None = None
        self.active_input: Path | None = None
        self.pending_input: Path | None = None
        self.active_input_hash: str | None = None
        self.pending_input_hash: str | None = None
        self.pending_ocr = False
        for name in ("inputs", "outputs", "models", "hf-cache", "resources"):
            self.child(name).mkdir(parents=True, exist_ok=True)
        self.python = self.child("venv") / "bin/python"

    def child(self, name: str) -> Path:
        """Keep all managed directories inside the dedicated demo directory."""
        path = self.root / name
        if path.is_symlink() or not path.resolve().is_relative_to(self.root):
            raise DemoError(f"Refusing a redirected demo path: {name}")
        return path

    @property
    def api_base(self) -> str:
        return f"http://127.0.0.1:{self.port}"

    def redact(self, value: str) -> str:
        for key in self._redaction_keys | ({self.key} if self.key else set()):
            value = value.replace(key, "[REDACTED]")
        return value

    def log_tail(self, name: str = "server.log", limit: int = 3000) -> str:
        path = self.child(name)
        if not path.is_file():
            return "No log available yet."
        with path.open("rb") as stream:
            stream.seek(max(0, path.stat().st_size - limit))
            return self.redact(stream.read(limit).decode("utf-8", errors="replace"))

    def prepare_sample(self, name: str, specification: dict) -> Path:
        """Fetch public sample data over HTTPS; validate it before saving or use."""
        types = {"parsecrate-demo.pdf": "application/pdf", "parsecrate-scan.png": "image/png"}
        url = specification.get("url")
        expected_hash = specification.get("sha256")
        size = specification.get("size_bytes")
        if (name not in types or url != f"https://parsecrate.com/demo/{name}"
                or not isinstance(expected_hash, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_hash)
                or type(size) is not int or not 0 < size <= 2 * 1024 * 1024):
            raise DemoError("Invalid sample URL, checksum, or size. Use the reviewed sample manifest.")
        target = self.child(f"resources/assets/{name}")
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.stat().st_size == size:
            with target.open("rb") as cached:
                content = cached.read(size + 1)
            if hashlib.sha256(content).hexdigest() == expected_hash:
                print(f"Using verified cached sample: {name}")
                return target
        # Discard an invalid managed cache so it cannot be mistaken for a sample.
        target.unlink(missing_ok=True)

        # HTTPSConnection verifies TLS, ignores environment proxies, and follows
        # no redirects. Public sample requests never carry the local API key.
        connection = http.client.HTTPSConnection("parsecrate.com", timeout=30)
        try:
            connection.request("GET", f"/demo/{name}", headers={"User-Agent": "ParseCrate-Colab-Demo/1.0"})
            response = connection.getresponse()
            if response.status != 200:
                raise DemoError(f"Sample download returned HTTP {response.status}: {url}. Retry later or contact the maintainer.")
            if response.getheader("Content-Type", "").split(";", 1)[0].lower() != types[name]:
                raise DemoError(f"The sample URL did not return the expected file type: {url}")
            content = response.read(size + 1)
        except (OSError, http.client.HTTPException) as exc:
            raise DemoError(f"Could not download {url}. Check your connection and retry setup.") from exc
        finally:
            connection.close()
        if len(content) != size or hashlib.sha256(content).hexdigest() != expected_hash:
            raise DemoError(f"Sample size or SHA-256 verification failed: {name}. The file was not accepted.")

        temporary = None
        try:
            with tempfile.NamedTemporaryFile(dir=target.parent, prefix=".sample-", delete=False) as output:
                temporary = Path(output.name)
                output.write(content)
            temporary.replace(target)
        except OSError as exc:
            raise DemoError("Could not save the verified sample. Check available runtime storage and retry setup.") from exc
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
        print(f"Downloaded and verified: {name} ({size:,} bytes)")
        return target

    @staticmethod
    def _terminate(process: subprocess.Popen) -> None:
        if process.poll() is not None:
            return
        # Every process handled here was started with start_new_session=True.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=10)

    def stop_server(self) -> None:
        if self.key:
            self._redaction_keys.add(self.key)
        if self.process is not None:
            self._terminate(self.process)
            self.process = None
        self.key = ""
        self.task_id = None
        self.submission_uncertain = False

    def _run(self, args: list[str], log: str, timeout: int, env=None) -> None:
        with self.child(log).open("ab") as output:
            proc = subprocess.Popen(
                args, cwd=self.root, env=env, stdout=output,
                stderr=subprocess.STDOUT, start_new_session=True,
            )
            try:
                code = proc.wait(timeout=timeout)
            except subprocess.TimeoutExpired as exc:
                self._terminate(proc)
                print(self.log_tail(log))
                raise DemoError(f"Command timed out after {timeout} seconds. See {log} above.") from exc
            except BaseException:
                self._terminate(proc)
                raise
        if code:
            print(self.log_tail(log))
            raise DemoError(f"Command failed. See {log} above.")

    def environment(self) -> dict[str, str]:
        # Avoid settings from a previous notebook or an unrelated .env file.
        prefixes = ("DOCLING_", "UVICORN_", "HF_", "HUGGINGFACE_", "OTEL_")
        env = {k: v for k, v in os.environ.items() if not k.startswith(prefixes)}
        for key in ("HUGGING_FACE_HUB_TOKEN", "PYTHONPATH", "PYTHONHOME"):
            env.pop(key, None)
        env.update({
            "HF_HOME": str(self.child("hf-cache")),
            "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
            "HF_HUB_DISABLE_TELEMETRY": "1",
            "DOCLING_DEVICE": "cpu",
            "DOCLING_NUM_THREADS": str(max(1, min(os.cpu_count() or 1, 4))),
        })
        return env

    def install(self) -> None:
        started = time.monotonic()
        self.stop_server()
        if platform.system() != "Linux" or platform.machine() != "x86_64":
            raise DemoError("Use a standard Google Colab Linux x86_64 runtime for this CPU demo.")
        if sys.version_info[:2] not in ((3, 12), (3, 13)):
            raise DemoError("This dependency baseline supports Python 3.12 and 3.13. Use a compatible Colab runtime.")
        requirements = self.child("resources") / "colab-requirements.txt"
        fingerprint = hashlib.sha256(
            requirements.read_bytes() + str(sys.version_info[:2]).encode()
        ).hexdigest()
        marker = self.child("venv-baseline.json")
        ready = self.python.is_file() and marker.is_file() and marker.read_text() == fingerprint
        # A cached environment is ready only after this run's checks also pass.
        # On failure or interruption, the next attempt must rebuild it.
        marker.unlink(missing_ok=True)
        # Target the demo interpreter explicitly to keep packages separate from
        # the notebook kernel.
        pip = [sys.executable, "-m", "pip", "--isolated", "--python", str(self.python),
               "--require-virtualenv", "--disable-pip-version-check"]
        if not ready:
            env_dir = self.child("venv")
            if env_dir.exists():
                shutil.rmtree(env_dir)
            print("Creating the isolated Python environment...")
            self._run([sys.executable, "-m", "venv", "--without-pip", str(env_dir)],
                      "install.log", 120, self.environment())
            print("Installing CPU PyTorch...")
            self._run(pip + ["install", "--only-binary=:all:", "--index-url", "https://download.pytorch.org/whl/cpu",
                           "torch==2.14.1+cpu", "torchvision==0.29.1+cpu"], "install.log", 1200, self.environment())
            print("Installing the document-processing dependencies...")
            self._run(pip + ["install", "--only-binary=:all:", "--index-url", "https://pypi.org/simple", "-r", str(requirements)],
                      "install.log", 1200, self.environment())
        else:
            print("Reusing the installed dependencies in this runtime.")
        print("Checking dependencies and CPU imports...")
        self._run(pip + ["check"], "install.log", 120, self.environment())
        probe = (
            "import json, importlib.metadata as m, torch, torchvision; "
            "from rapidocr import RapidOCR; import docling_serve.app; "
            "assert torch.version.cuda is None, 'Expected CPU-only PyTorch'; "
            "assert torch.ones(2).sum().item() == 2; "
            "print(json.dumps({n: m.version(n) for n in "
            "['docling-serve','docling-slim','docling-core','docling-jobkit','torch','torchvision','rapidocr']}))"
        )
        self._run([str(self.python), "-c", probe], "install.log", 180, self.environment())
        marker.write_text(fingerprint)
        print(f"Dependencies ready in {time.monotonic() - started:.1f}s. Details: install.log")

    def prepare_models(self) -> None:
        started = time.monotonic()
        print("Preparing layout, table, and RapidOCR models (cached on subsequent runs)...")
        self._run(
            [str(self.python.parent / "docling-tools"), "models", "download", "-o",
             str(self.child("models")), "--rapidocr-backend-lang", "onnxruntime:ch",
             "--rapidocr-backend-lang", "onnxruntime:en", "layout", "tableformer", "rapidocr"],
            "models.log", 1200, self.environment(),
        )
        print(f"Models ready in {time.monotonic() - started:.1f}s.")

    def _request(self, method: str, path: str, body: bytes | None = None,
                 headers: dict | None = None, timeout: float = 30,
                 require_success: bool = True) -> tuple[int, dict, bytes]:
        if not path.startswith("/") or path.startswith("//") or "\r" in path or "\n" in path:
            raise DemoError("Invalid local API path.")
        # http.client neither follows redirects nor uses environment proxies.
        conn = http.client.HTTPConnection("127.0.0.1", self.port, timeout=timeout)
        try:
            conn.request(method, path, body=body, headers=headers or {})
            reply = conn.getresponse()
            data = reply.read(self.MAX_RESPONSE_BYTES + 1)
            if len(data) > self.MAX_RESPONSE_BYTES:
                raise DemoError("API response exceeded the demo's 32 MiB limit.")
            status = reply.status
            response_headers = {k.lower(): v for k, v in reply.getheaders()}
        except (OSError, http.client.HTTPException) as exc:
            raise DemoError("Local API connection failed. Check server logs; an accepted task may still be running.") from exc
        finally:
            conn.close()
        if require_success and not 200 <= status < 300:
            detail = self.redact(data.decode("utf-8", errors="replace")[:600])
            raise DemoError(f"API returned HTTP {status}: {detail}")
        return status, response_headers, data

    @staticmethod
    def _json(headers: dict, data: bytes) -> dict:
        if "application/json" not in headers.get("content-type", "").lower():
            raise DemoError("Expected a JSON API response, not an archive or HTML page.")
        try:
            value = json.loads(data)
        except (ValueError, UnicodeError) as exc:
            raise DemoError("The API returned invalid JSON.") from exc
        if not isinstance(value, dict):
            raise DemoError("The API returned an unexpected JSON structure.")
        return value

    def _auth(self) -> dict[str, str]:
        if not self.key:
            raise DemoError("Start the API server before running this cell.")
        return {"X-Api-Key": self.key}

    def start_server(self, deadline_seconds: float = 600) -> str:
        self.stop_server()
        self.active_output = None
        self.active_response = None
        self.active_input = None
        with socket.socket() as probe:
            # Match Uvicorn's socket reuse and avoid false conflicts from TIME_WAIT.
            probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                probe.bind(("127.0.0.1", self.port))
            except OSError as exc:
                raise DemoError(f"Port {self.port} is occupied. Choose another port; unrelated processes were left alone.") from exc
        scratch = self.child("server-scratch")
        scratch.mkdir(exist_ok=True)
        self.key = secrets.token_urlsafe(32)
        self._redaction_keys.add(self.key)
        env = self.environment()
        env.update({
            "UVICORN_HOST": "127.0.0.1", "UVICORN_PORT": str(self.port),
            "UVICORN_WORKERS": "1", "UVICORN_RELOAD": "false",
            "DOCLING_SERVE_API_KEY": self.key,
            "DOCLING_SERVE_ARTIFACTS_PATH": str(self.child("models")),
            "DOCLING_SERVE_SCRATCH_PATH": str(scratch),
            "DOCLING_SERVE_ENABLE_UI": "false",
            "DOCLING_SERVE_ENABLE_REMOTE_SERVICES": "false",
            "DOCLING_SERVE_LOAD_MODELS_AT_BOOT": "true",
            "DOCLING_SERVE_ENG_KIND": "local",
            "DOCLING_SERVE_ENG_LOC_NUM_WORKERS": "1",
            "DOCLING_SERVE_OPTIONS_CACHE_SIZE": "1",
            "DOCLING_SERVE_DEFAULT_OCR_PRESET": "rapidocr",
            "DOCLING_SERVE_DEFAULT_OCR_KIND": "rapidocr",
            "DOCLING_SERVE_MAX_FILE_SIZE": str(self.MAX_FILE_BYTES),
            "DOCLING_SERVE_MAX_NUM_PAGES": "10",
            "DOCLING_SERVE_MAX_DOCUMENT_TIMEOUT": "600",
            "DOCLING_SERVE_MAX_SYNC_WAIT": "120",
            "HF_HUB_OFFLINE": "1",
        })
        try:
            with self.child("server.log").open("wb") as output:
                self.process = subprocess.Popen(
                    [str(self.python.parent / "docling-serve"), "run"], cwd=self.root,
                    env=env, stdout=output, stderr=subprocess.STDOUT, start_new_session=True,
                )
            deadline = time.monotonic() + deadline_seconds
            print("Starting the local API and loading models...")
            while time.monotonic() < deadline:
                if self.process.poll() is not None:
                    raise DemoError("The server exited during startup.")
                try:
                    status, _, _ = self._request("GET", "/ready", timeout=3, require_success=False)
                    if status == 200:
                        break
                except DemoError:
                    pass
                time.sleep(2)
            else:
                raise DemoError("The API did not become ready within the startup deadline.")
            self.verify_auth()
            print(f"API ready at {self.api_base}. CPU processing and X-Api-Key authentication enabled.")
            return self.key
        except BaseException:
            print(self.log_tail())
            self.stop_server()
            raise

    def _multipart(self, path: Path, fields: list[tuple[str, str]]) -> tuple[bytes, str, str]:
        path = Path(path)
        content = self._read_input(path)
        boundary = "ParseCrate" + secrets.token_hex(16)
        chunks = []
        for name, value in fields:
            chunks.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"\r\n\r\n{value}\r\n'.encode())
        # A neutral filename avoids embedding an uploaded filename in headers.
        suffix = path.suffix.lower()
        suffix = suffix if re.fullmatch(r"\.[a-z0-9]{1,12}", suffix) else ".bin"
        mime = mimetypes.guess_type("document" + suffix)[0] or "application/octet-stream"
        chunks.append(f'--{boundary}\r\nContent-Disposition: form-data; name="files"; filename="document{suffix}"\r\nContent-Type: {mime}\r\n\r\n'.encode())
        chunks.extend([content, f"\r\n--{boundary}--\r\n".encode()])
        return b"".join(chunks), f"multipart/form-data; boundary={boundary}", hashlib.sha256(content).hexdigest()

    def _read_input(self, path: Path) -> bytes:
        if not path.is_file() or not 0 < path.stat().st_size <= self.MAX_FILE_BYTES:
            raise DemoError("Choose a nonempty file no larger than 20 MiB.")
        with path.open("rb") as stream:
            content = stream.read(self.MAX_FILE_BYTES + 1)
        if not 0 < len(content) <= self.MAX_FILE_BYTES:
            raise DemoError("The input file changed or exceeded the 20 MiB demo limit.")
        return content

    def verify_auth(self) -> None:
        sample = self.child("inputs") / "parsecrate-demo.pdf"
        body, content_type, _ = self._multipart(sample, [("to_formats", "md")])
        for headers in ({}, {"X-Api-Key": self.key + "-incorrect"}):
            status, _, _ = self._request(
                "POST", "/v1/convert/file/async", body,
                {"Content-Type": content_type, **headers}, require_success=False,
            )
            if status != 401:
                raise DemoError(f"Authentication check failed: expected HTTP 401, got {status}.")

    def _begin_submission(self) -> None:
        if self.submission_uncertain:
            raise DemoError("The previous submission had an uncertain outcome. Inspect the server or restart it before submitting again.")
        if self.task_id:
            raise DemoError("A task is already recorded. Resume it in section 6 or restart the server before submitting again.")
        self.active_output = None
        self.active_response = None
        self.active_input = None
        self.active_input_hash = None

    def submit(self, path: Path, *, kind: str = "convert", ocr: bool = False) -> str:
        self._begin_submission()
        if kind not in ("convert", "chunk"):
            raise DemoError("Unknown demo operation.")
        fields = [("target_type", "inbody")]
        if kind == "convert":
            fields += [("to_formats", "md"), ("to_formats", "json"),
                       ("image_export_mode", "placeholder"),
                       ("do_ocr", str(ocr).lower()), ("ocr_preset", "rapidocr")]
            if ocr:
                fields.append(("ocr_lang", "en"))
            endpoint = "/v1/convert/file/async"
        else:
            fields += [("convert_image_export_mode", "placeholder"),
                       ("convert_do_ocr", "false"), ("include_converted_doc", "false")]
            endpoint = "/v1/chunk/hierarchical/file/async"
        body, content_type, source_hash = self._multipart(path, fields)
        auth = self._auth()
        self.pending_input = Path(path).resolve()
        self.pending_input_hash = source_hash
        self.pending_ocr = ocr
        self.submission_uncertain = True
        _, headers, data = self._request("POST", endpoint, body,
                                         {"Content-Type": content_type, **auth})
        payload = self._json(headers, data)
        task_id = payload.get("task_id")
        if not isinstance(task_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", task_id):
            raise DemoError("The submission returned no valid task ID. Inspect the server before submitting again.")
        self.task_id, self.task_kind = task_id, kind
        self.submission_uncertain = False
        (self.child("outputs") / "last-task.json").write_text(json.dumps({"task_id": task_id, "kind": kind}))
        print(f"Task accepted: {task_id}. Resume polling this task after an interruption; do not resubmit it automatically.")
        return task_id

    def wait_result(self, timeout: float = 600, interval: float = 2) -> dict:
        if not self.task_id:
            raise DemoError("No task to resume. Submit one document first.")
        task_id = self.task_id
        deadline = time.monotonic() + timeout
        previous = None
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            _, headers, data = self._request(
                "GET", f"/v1/status/poll/{task_id}", headers=self._auth(),
                timeout=max(0.1, min(30, remaining)),
            )
            task = self._json(headers, data)
            state = task.get("task_status")
            if not isinstance(state, str) or state not in self.PENDING | self.TERMINAL:
                raise DemoError("Unknown task status. Check the installed API/schema version.")
            if state != previous:
                print("Task status:", state)
                previous = state
            if state in {"failure", "skipped"}:
                detail = self.redact(str(task.get("error_message") or "See the server log.")[:600])
                self.task_id = None
                raise DemoError(f"Document task ended with {state}: {detail}")
            if state in {"success", "partial_success"}:
                break
            time.sleep(min(interval, max(0, deadline - time.monotonic())))
        else:
            raise DemoError("Polling deadline reached. The task ID is retained; rerun the resume cell, or stop the server to abandon it.")

        _, headers, data = self._request("GET", f"/v1/result/{task_id}", headers=self._auth())
        # Persist before parsing: results may be consumed after their first fetch.
        output = self._persist_response(task_id, data)
        self.task_id = None
        return self._activate_result(output, headers, data, kind=self.task_kind,
                                     source=self.pending_input, source_hash=self.pending_input_hash,
                                     state=state)

    def _persist_response(self, name: str, data: bytes) -> Path:
        output = self.child(f"outputs/{name}")
        output.mkdir(exist_ok=True)
        (output / "conversion-response.json").write_bytes(data)
        return output

    def _activate_result(self, output: Path, headers: dict, data: bytes, *, kind: str,
                         source: Path | None, source_hash: str | None, state: str = "") -> dict:
        payload = self._json(headers, data)
        if kind == "chunk":
            if not isinstance(payload.get("chunks"), list) or not payload["chunks"]:
                raise DemoError("The chunk response contained no chunks. The response was saved.")
            (output / "chunks.json").write_text(json.dumps(payload["chunks"], ensure_ascii=False, indent=2))
        else:
            doc = payload.get("document")
            if not isinstance(doc, dict):
                raise DemoError("No document in the saved conversion response.")
            markdown, structured = doc.get("md_content"), doc.get("json_content")
            status = payload.get("status")
            if not isinstance(status, str) or status not in {"success", "partial_success"}:
                raise DemoError("The conversion did not succeed. Inspect the saved response and logs.")
            if not isinstance(markdown, str) or not markdown.strip() or not isinstance(structured, dict) or not structured:
                raise DemoError("Requested Markdown/structured JSON is empty. The API response was saved for inspection.")
            (output / "converted.md").write_text(markdown, encoding="utf-8")
            (output / "converted.json").write_text(json.dumps(structured, ensure_ascii=False, indent=2), encoding="utf-8")
        self.active_output, self.active_response = output, payload
        self.active_input = source
        self.active_input_hash = source_hash
        if state == "partial_success" or payload.get("status") == "partial_success" or payload.get("errors"):
            print("Partial result or conversion warnings: review the saved errors before using this output.")
        else:
            print("Conversion complete.")
        print("Saved outputs:", output)
        return payload

    def convert(self, path: Path, *, kind: str = "convert", ocr: bool = False) -> dict:
        if self.task_id:
            if (kind != self.task_kind or ocr != self.pending_ocr
                    or Path(path).resolve() != self.pending_input
                    or hashlib.sha256(self._read_input(Path(path))).hexdigest() != self.pending_input_hash):
                raise DemoError("A different task is recorded. Resume it in section 6 before starting another example.")
            print("Resuming the recorded task without submitting it again.")
        else:
            self.submit(path, kind=kind, ocr=ocr)
        return self.wait_result()

    def has_saved_conversion(self, path: Path) -> bool:
        return (self.active_response is not None and "document" in self.active_response
                and self.active_input == Path(path).resolve() and Path(path).is_file()
                and Path(path).stat().st_size <= self.MAX_FILE_BYTES
                and hashlib.sha256(self._read_input(Path(path))).hexdigest() == self.active_input_hash)

    def synchronous_example(self, path: Path) -> dict:
        self._begin_submission()
        body, content_type, source_hash = self._multipart(path, [("target_type", "inbody"),
            ("to_formats", "md"), ("to_formats", "json"), ("do_ocr", "false")])
        auth = self._auth()
        self.submission_uncertain = True
        _, headers, data = self._request("POST", "/v1/convert/file", body,
            {"Content-Type": content_type, **auth}, timeout=150)
        output = self._persist_response("sync-" + secrets.token_hex(8), data)
        self.submission_uncertain = False
        return self._activate_result(output, headers, data, kind="convert",
                                     source=Path(path).resolve(), source_hash=source_hash)

    def save_upload(self, uploaded: dict[str, bytes], previous: Path) -> Path:
        if not uploaded:
            print("Upload cancelled; keeping the previous input.")
            return previous
        if len(uploaded) != 1:
            raise DemoError("Select one document at a time, then run the upload cell again.")
        name, content = next(iter(uploaded.items()))
        suffix = Path(name.replace("\\", "/")).suffix.lower()
        if suffix not in {".pdf", ".docx"}:
            raise DemoError("Use PDF or DOCX here. The separate OCR cell demonstrates an image input.")
        if not 0 < len(content) <= self.MAX_FILE_BYTES:
            raise DemoError("Choose a nonempty file no larger than 20 MiB.")
        path = self.child("inputs") / ("upload-" + secrets.token_hex(6) + suffix)
        path.write_bytes(content)
        return path

    def archive_results(self) -> Path:
        if self.active_output is None:
            raise DemoError("Complete a conversion first.")
        target = self.child("outputs") / (self.active_output.name + "-download")
        return Path(shutil.make_archive(str(target), "zip", self.active_output))

In [ ]:
# @title Load dependency requirements { display-mode: "form" }
PINNED_REQUIREMENTS = """# ParseCrate Colab CPU baseline, resolved October 18, 2026.
# Linux x86_64, Python 3.12/3.13. CPU torch wheels are installed first.
# All transitive dependencies are pinned.
accelerate==1.15.0
agent-detector==2.0.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiohttp-cors==0.8.1
aiohttp-retry==2.9.1
aiosignal==1.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
asgiref==3.12.1
asttokens==3.0.2
attrs==26.1.0
av==19.0.1
azure-core==1.41.0
azure-storage-blob==12.31.0
bcrypt==5.0.0
beautifulsoup4==4.15.0
boto3==1.43.114
botocore==1.43.114
certifi==2026.10.18
cffi==2.1.1
charset-normalizer==3.5.2
click==8.5.0
codeflare-sdk==0.40.1
colorful==0.5.8
colorlog==6.12.0
comm==0.2.3
croniter==6.2.4
cryptography==50.0.2
defusedxml==0.7.1
detect-installer==0.2.1
dill==0.4.1
distlib==0.4.4
dnspython==2.9.0
doclang==0.7.3
docling-core==2.102.0
docling-ibm-models==4.0.3
docling-jobkit==3.8.2
docling-mcp==3.3.0
docling-parse==7.22.3
docling-serve==1.37.0
docling-slim==2.140.0
durationpy==0.11
einops==0.8.2
email-validator==2.3.0
et-xmlfile==2.0.0
executing==2.2.1
faker==40.43.0
fastapi==0.141.1
fastapi-cli==0.0.32
fastapi-cloud-cli==0.26.0
fastar==0.12.0
filelock==4.2.4
filetype==1.2.0
flatbuffers==25.12.19
frozenlist==1.8.0
fsspec==2026.9.0
ftfy==6.3.1
google-api-core==2.34.0
google-api-python-client==2.201.0
google-auth==2.62.0
google-auth-httplib2==0.4.4
google-auth-oauthlib==1.5.0
google-cloud-core==2.8.0
google-cloud-storage==3.17.1
google-crc32c==1.9.0
google-resumable-media==2.11.0
googleapis-common-protos==1.75.5
grpcio==1.84.0
h11==0.16.0
hf-xet==1.7.0
hiredis==3.5.0
httpcore==1.0.9
httpcore2==2.14.0
httplib2==0.32.0
httptools==0.9.0
httpx==0.28.1
httpx2==2.14.0
huggingface-hub==1.33.0
idna==3.20
importlib-metadata==8.7.1
invoke==3.0.3
ipython==9.17.1
ipython-pygments-lexers==1.1.1
ipywidgets==8.1.2
isodate==0.7.2
jedi==0.20.1
jinja2==3.1.6
jmespath==1.1.0
jsonref==1.1.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyterlab-widgets==3.0.17
kube-authkit==0.4.0
kubernetes==37.0.1
langcodes==3.5.1
latex2mathml==3.81.1
lxml==6.1.3
mail-parser==4.8.0
markdown-it-py==4.2.0
marko==2.2.4
markupsafe==3.0.4
matplotlib-inline==0.2.2
mcp==2.3.0
mcp-types==2.3.0
mdurl==0.1.2
ml-dtypes==0.5.4
mmh3==5.3.1
mpire==2.10.2
mpmath==1.3.0
msgpack==1.2.3
multidict==7.1.0
multiprocess==0.70.19
networkx==3.7
numpy==2.5.4
oauthlib==4.0.0
odfdo==3.27.2
olefile==0.47
omegaconf==2.4.0
onnxruntime==1.31.0
open-clip-torch==3.3.0
opencensus==0.11.4
opencensus-context==0.1.3
opencv-python-headless==5.0.0.93
openpyxl==3.1.5
openshift-client==1.0.18
opentelemetry-api==1.36.0
opentelemetry-exporter-otlp==1.36.0
opentelemetry-exporter-otlp-proto-common==1.36.0
opentelemetry-exporter-otlp-proto-grpc==1.36.0
opentelemetry-exporter-otlp-proto-http==1.36.0
opentelemetry-exporter-prometheus==0.57b0
opentelemetry-instrumentation==0.57b0
opentelemetry-instrumentation-asgi==0.57b0
opentelemetry-instrumentation-fastapi==0.57b0
opentelemetry-proto==1.36.0
opentelemetry-sdk==1.36.0
opentelemetry-semantic-conventions==0.57b0
opentelemetry-util-http==0.57b0
packaging==26.3
pandas==2.3.3
paramiko==5.1.0
parso==0.8.7
peft==0.21.2
pexpect==4.9.0
pillow==12.3.0
platformdirs==4.13.5
pluggy==1.7.0
polyfactory==3.3.0
prometheus-client==0.26.0
prompt-toolkit==3.0.53
propcache==0.5.4
proto-plus==1.29.0
protobuf==6.33.6
psutil==7.2.2
ptyprocess==0.7.0
pure-eval==0.2.4
py-spy==0.4.2
pyarrow==26.0.0
pyasn1==0.6.4
pyasn1-modules==0.4.2
pyclipper==1.4.0
pycparser==3.11
pydantic==2.14.1
pydantic-core==2.50.1
pydantic-extra-types==2.11.1
pydantic-settings==2.16.0
pygments==2.21.0
pyjwt==2.15.1
pylatexenc==2.11
pynacl==1.6.2
pyparsing==3.3.3
pypdfium2==5.14.0
python-dateutil==2.9.0.post0
python-discovery==1.6.4
python-docx==1.2.0
python-dotenv==1.2.4
python-multipart==0.0.32
python-oxmsg==0.0.2
python-pptx==1.0.2
python-rapidjson==1.25
pytz==2026.5
pyyaml==6.0.3
qwen-vl-utils==0.0.14
rapidocr==3.10.0
ray==2.58.0
ray-haproxy==2.8.25
redis==7.4.1
referencing==0.37.0
regex==2026.9.29
requests==2.34.2
requests-oauthlib==2.0.0
rich==14.3.4
rich-toolkit==0.20.6
rignore==0.8.1
rpds-py==2026.9.1
rq==2.12.0
rtree==1.4.1
s3transfer==0.19.2
safetensors==0.8.0
scalar-fastapi==1.9.1
scipy==1.18.1
semchunk==3.2.5
sentry-sdk==2.72.0
setuptools==84.0.0
shapely==2.2.0
shellingham==1.5.4
six==1.17.0
smart-open==8.0.4
soupsieve==3.0.3
sse-starlette==3.5.0
stack-data==0.6.3
starlette==1.9.0
sympy==1.14.0
tabulate==0.10.0
timm==1.0.30
tokenizers==0.23.3
torch==2.14.1+cpu
torchvision==0.29.1+cpu
tqdm==4.70.1
traitlets==5.16.1
transformers==5.20.1
tree-sitter==0.26.0
tree-sitter-c==0.24.2
tree-sitter-javascript==0.25.0
tree-sitter-python==0.25.0
tree-sitter-typescript==0.23.2
tritonclient==2.73.0
truststore==0.10.5
typer==0.26.8
typing-extensions==4.16.0
typing-inspection==0.4.4
tzdata==2026.5
uritemplate==4.2.0
urllib3==2.8.0
uvicorn==0.54.0
uvloop==0.23.0
virtualenv==21.14.11
watchfiles==1.3.0
wcwidth==0.9.2
websocket-client==1.9.2
websockets==16.1.1
widgetsnbextension==4.0.16
wrapt==1.17.3
xlsxwriter==3.2.9
yarl==1.25.1
zipp==4.1.1
"""

In [ ]:
SAMPLE_FILES = {
    "parsecrate-demo.pdf": {
        "url": "https://parsecrate.com/demo/parsecrate-demo.pdf",
        "size_bytes": 2540,
        "sha256": "aa49a5b14c7f1a49ad27e2f1fb36a80eab66cee88daaf4075ed161b740b7efec"
    },
    "parsecrate-scan.png": {
        "url": "https://parsecrate.com/demo/parsecrate-scan.png",
        "size_bytes": 108254,
        "sha256": "ae9103c2b4938450d11cc07339401be245f1cddd35b157e3340495c044eec48f"
    }
}

# Stop the previous demo server before resetting setup.
if globals().get("parse_demo") is not None:
    parse_demo.stop_server()
parse_api_key = None
sample_path = selected_path = None
parse_demo = ParseCrateDemo(DEMO_ROOT, API_PORT)

requirements_path = parse_demo.child("resources/colab-requirements.txt")
requirements_path.write_text(PINNED_REQUIREMENTS, encoding="utf-8")
for name, specification in SAMPLE_FILES.items():
    parse_demo.prepare_sample(name, specification)
sample_path = parse_demo.child("inputs/parsecrate-demo.pdf")
sample_path.write_bytes(parse_demo.child("resources/assets/parsecrate-demo.pdf").read_bytes())
selected_path = sample_path
print("Samples verified. CPU mode; 10-page and 20 MiB document limits.")

## 2. Install the API dependencies

This installs the CPU dependencies in a separate Python environment,
leaving Colab's existing packages unchanged.
The first run downloads PyTorch and the document-processing packages
and can take several minutes. Each setup stage is reported below;
installation failures show the relevant log.
Successful installations are reused on later runs. If validation
fails, rerun this cell to rebuild the environment.

In [ ]:
parse_demo.install()

## 3. Prepare local models

ParseCrate's Docker image includes default models. This Python demo
downloads layout, table, and RapidOCR models into its own cache.
Cached files are reused within this runtime. A runtime reset removes
them. No Google Drive mount or model-provider token is needed.

In [ ]:
parse_demo.prepare_models()

## 4. Start the authenticated local API

The server listens only on `127.0.0.1`, loads the prepared models,
and checks that missing/incorrect keys are rejected. A temporary
random key is generated for this server and kept in memory.

In [ ]:
parse_api_key = None
parse_api_key = parse_demo.start_server()

## 5. Choose your document

The original one-page PDF includes headings, a checklist, and a
small table. It is used automatically during Run all.

To try your own **PDF or DOCX**, set `UPLOAD_OWN_DOCUMENT = True` in
the next cell and run it manually. Then rerun sections 6 and 7.
Select one file, at most **20 MiB and 10 pages**. Larger documents
may fail or return partial output; review the reported status.

In [ ]:
UPLOAD_OWN_DOCUMENT = False

if UPLOAD_OWN_DOCUMENT:
    import tempfile
    from google.colab import files as colab_files
    # Remove temporary upload copies after validating the document.
    with tempfile.TemporaryDirectory(prefix="incoming-", dir=DEMO_ROOT / "inputs") as incoming:
        selected_path = parse_demo.save_upload(colab_files.upload(target_dir=incoming), selected_path)
print(f"Input: {selected_path.name} ({selected_path.stat().st_size:,} bytes)")

## 6. Convert the document

The file is submitted to `POST /v1/convert/file/async` with the
`X-Api-Key` header and both `md` and `json` output formats. The
notebook polls status, then saves the result from
`GET /v1/result/{task_id}`. This avoids a long synchronous request.

If polling is interrupted, rerun this cell to **resume the recorded
task**. It will not submit that known task again. If the initial
submission had an uncertain outcome, inspect the logs or restart
the server before retrying. A client timeout does not cancel work
already accepted by the API.

A completed conversion of the same unchanged file is reused. Set
`FORCE_NEW_CONVERSION = True` only when you want to convert it again.

In [ ]:
FORCE_NEW_CONVERSION = False

if parse_demo.task_id:
    result = parse_demo.wait_result()
elif not FORCE_NEW_CONVERSION and parse_demo.has_saved_conversion(selected_path):
    result = parse_demo.active_response
    print("Reusing the saved conversion. No new API task was submitted.")
else:
    result = parse_demo.convert(selected_path, ocr=selected_path.suffix.lower() == ".png")
if "document" in result and parse_demo.active_input_hash == SAMPLE_FILES["parsecrate-demo.pdf"]["sha256"]:
    text = " ".join(result["document"]["md_content"].casefold().split())
    if not all(word in text for word in ("cedar", "workshop", "casey", "morgan", "riley")):
        raise RuntimeError("Sample text verification failed. Inspect the saved response and preview.")
    if not result["document"]["json_content"].get("tables"):
        raise RuntimeError("The sample table was not identified in structured JSON. Inspect the output.")
    print("Sample text and structured table checks passed.")
# Display cells use the saved response, without refetching the task.

## 7. Inspect the extracted content

The sample's Markdown preview shows headings and its table. Expand
**View structured JSON** for a short preview of the document structure.
Uploaded documents are previewed as escaped text to avoid loading
document-supplied HTML or external resources.

Outputs are saved under a separate run folder as `converted.md`,
`converted.json`, and `conversion-response.json`. Download/display
cells reuse these files because server results may expire after a
fetch. Review any partial-success status or warnings.

In [ ]:
from html import escape
from IPython.display import Code, HTML, Markdown, display

result = parse_demo.active_response
if result is None:
    raise RuntimeError("No current saved result. Complete a conversion first.")
summary = [f"Status: {result.get('status', 'chunk result')}"]
elapsed = result.get("processing_time")
if type(elapsed) in (int, float):
    summary.append(f"Time: {elapsed:.1f}s")
if "document" in result:
    document = result["document"]
    structured = document["json_content"]
    if isinstance(structured.get("pages"), (dict, list)):
        summary.append(f"Pages: {len(structured['pages'])}")
    if isinstance(structured.get("tables"), list):
        summary.append(f"Tables: {len(structured['tables'])}")
print(" · ".join(summary))
print("Saved files:", parse_demo.active_output)
if "document" in result:
    preview = json.dumps(structured, ensure_ascii=False, indent=2)
    if len(preview) > 2500:
        preview = preview[:2500] + "\n… Preview truncated. Download converted.json for the full document."
    display(HTML(
        "<details><summary>View structured JSON</summary>"
        '<pre style="max-height:16rem;overflow:auto;white-space:pre-wrap">'
        + escape(preview) + "</pre></details>"
    ))
    markdown = document["md_content"]
    if parse_demo.active_input_hash == SAMPLE_FILES["parsecrate-demo.pdf"]["sha256"]:
        display(Markdown(markdown))
    else:
        display(Code(markdown[:12000], language="markdown"))
else:
    display(Code(json.dumps(result["chunks"][:3], ensure_ascii=False, indent=2)[:5000], language="json"))

### Optional: download the saved files

Set the flag below to download one ZIP. Run all leaves downloads off.

In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    from google.colab import files as colab_files
    colab_files.download(str(parse_demo.archive_results()))

## 8. Optional API examples

These examples are disabled during Run all. Enable and run one at a
time after the main conversion succeeds. They use the same local
API key. Each conversion saves its own output folder; the download
cell exports the most recent successful result.

If OCR or chunking polling is interrupted, rerun the same example
cell to resume its recorded task. Finish that task before starting
a different example.

### Synchronous conversion

`POST /v1/convert/file` is useful for small, quick jobs. This example
permits 150 seconds on the client with a 120-second server wait.
A timeout is reported without automatically resubmitting the file.
If its outcome is uncertain, inspect the logs and restart the
server before retrying. Successful or partial outputs use the same
saved-result preview and download cells above.

In [ ]:
RUN_SYNC_EXAMPLE = False

if RUN_SYNC_EXAMPLE:
    sync_result = parse_demo.synchronous_example(sample_path)
    print("Synchronous sample status:", sync_result["status"])

### Read a scanned page with OCR

The downloaded PNG is a rasterized copy of the original sample with no
text layer. This uses local RapidOCR models for its English text.
Re-running the results cell after this example shows the saved OCR
output. You can download it using the same download cell above.

In [ ]:
RUN_OCR_EXAMPLE = False

if RUN_OCR_EXAMPLE:
    from IPython.display import Code, Image, display
    scan_path = DEMO_ROOT / "resources/assets/parsecrate-scan.png"
    display(Image(filename=str(scan_path), width=500))
    ocr_result = parse_demo.convert(scan_path, ocr=True)
    selected_path = scan_path
    display(Code(ocr_result["document"]["md_content"][:12000], language="markdown"))

### Create hierarchical chunks

This submits the sample to
`POST /v1/chunk/hierarchical/file/async` and shows several chunks
with their available metadata. Hierarchical chunking needs no extra
embedding model. Connect the resulting text to your own retrieval
application, storage, and embeddings service as needed.

In [ ]:
RUN_CHUNK_EXAMPLE = False

if RUN_CHUNK_EXAMPLE:
    from IPython.display import Code, display
    chunk_result = parse_demo.convert(sample_path, kind="chunk")
    print("Chunks:", len(chunk_result["chunks"]))
    display(Code(json.dumps(chunk_result["chunks"][:3], ensure_ascii=False, indent=2)[:5000], language="json"))

## 9. Troubleshooting and cleanup

If setup fails, read the reported log excerpt. `install.log` and
`models.log` are separate from `server.log` under the demo directory.
Installation/model caches are reused after a successful setup.

- **Port occupied:** stop this notebook's server or select another
  `API_PORT`, then rerun from section 1. Unrelated processes are left alone.
- **Sample download failed:** check the URL and your connection, then
  rerun the sample-preparation cell. Invalid files are never used.
- **Polling deadline:** rerun section 6 to resume the known task.
- **Conversion error:** inspect the saved response and log, and try a
  small unencrypted document within the demo limits.
- **No free disk/RAM:** disconnect and delete the runtime, then try a
  fresh standard CPU runtime and the small sample.

The log cell below shows a bounded excerpt with the demo key redacted.

In [ ]:
print(parse_demo.log_tail())

### Stop the API server when finished

Set `STOP_SERVER = True` and run the cell. Saved results remain
available, so they can still be downloaded. To process another file,
rerun the start-server cell first. Run all leaves the server available
for your own experiments.

Use **Runtime → Disconnect and delete runtime** to end the session.
Download results you want to keep first. Runtime files and model
caches are temporary. Before sharing a saved notebook, clear outputs
so uploaded document contents are not included.

In [ ]:
STOP_SERVER = False

if STOP_SERVER:
    parse_demo.stop_server()
    parse_api_key = None
    print("Server stopped. Saved result files remain in the outputs directory.")

## Run ParseCrate on your own infrastructure

- [ParseCrate setup and API documentation](https://github.com/hwdsl2/parsecrate)
- [ParseCrate website](https://parsecrate.com/)
- [Self-Hosted AI Stack](https://selfhostedaistack.com/)
- [The Self-Hosted AI Builder's Guide](https://selfhostedaistack.com/book)

The original sample files and notebook support code are released under
the repository's MIT License. Docling, docling-serve, models, and
dependencies retain their own licenses. Google and download providers
have their own data practices.